In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')
model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3-mini-4k-instruct')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [ ]:
model.model.layers[0].self_attn.o_proj.weight.shape

torch.Size([3072, 3072])

In [ ]:
import torch

In [ ]:
A = torch.randn(3072, 3)
B = torch.randn(3, 3072)

In [ ]:
A @ B

tensor([[ 0.1316,  1.2824,  0.3052,  ...,  2.5804, -0.3659, -0.7240],
        [-0.6170,  3.7123,  0.9044,  ...,  8.4137, -0.3463,  0.0631],
        [-0.5580,  1.2437, -0.4472,  ...,  6.4437,  0.2703,  0.5250],
        ...,
        [-0.9221, -3.2774, -2.9478,  ...,  3.4346,  1.7433,  2.3807],
        [ 0.3134, -2.3787, -1.2983,  ..., -2.1818,  0.4093, -0.1066],
        [-0.2609,  0.9327,  0.8100,  ..., -0.3093, -0.1152,  0.4410]])

In [ ]:
import torch.nn as nn
import torch

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=5, alpha=1.0, bias=False):
        super().__init__()
        # Standard weight is (out_features, in_features)
        self.weight = nn.Parameter(torch.randn(out_features, in_features), requires_grad=False)
        self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False) if bias else None

        # LoRA matrices
        # A: (r, in_features), B: (out_features, r)
        # So that (B @ A) results in (out_features, in_features)
        self.A = nn.Parameter(torch.randn(r, in_features))
        self.B = nn.Parameter(torch.zeros(out_features, r))

        self.r = r
        self.alpha = alpha

    def forward(self, X):
        # base: (batch, out_features)
        base = X @ self.weight.T

        # lora: X (batch, in) @ A.T (in, r) @ B.T (r, out) -> (batch, out)
        lora = (X @ self.A.T) @ self.B.T

        out = base + (self.alpha / self.r) * lora
        if self.bias is not None:
            out += self.bias
        return out

# Re-initialize
fc = LoRALinear(100, 100, r=5)

In [ ]:
X = torch.randn(50, 100)

In [ ]:
fc(X)

tensor([[  2.1704,   9.8038,  -5.4668,  ...,  17.9446,  23.2482,  -3.8204],
        [ -3.2629,  -1.3923,   7.6180,  ..., -13.9889,   7.4269,   6.1552],
        [ -1.1125,  -5.1506,   6.9055,  ...,  -3.9222,  12.9964,  -9.7055],
        ...,
        [ 16.2905,  -8.6892,   0.9959,  ...,  -5.2014,   6.6963,  -6.7354],
        [ -5.4549,   3.6165,  -4.4789,  ...,   0.8561, -13.0904, -25.2091],
        [ 14.3133,  -7.6266,   3.6787,  ...,  15.1457,  38.5109,  -2.4935]],
       grad_fn=<AddBackward0>)

In [ ]:
class MyModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config['layer1'], config['layer2'])
        self.fc2 = nn.Linear(config['layer2'], config['layer3'])
        self.fc3 = nn.Linear(config['layer3'], config['layer4'])

    def forward(self, X):
        X = self.fc1(X)
        X = self.fc2(X)
        X = self.fc3(X)
        return X

model = MyModel(config={'layer1': 100, 'layer2': 150, 'layer3': 200, 'layer4': 250})
model

MyModel(
  (fc1): Linear(in_features=100, out_features=150, bias=True)
  (fc2): Linear(in_features=150, out_features=200, bias=True)
  (fc3): Linear(in_features=200, out_features=250, bias=True)
)

In [ ]:
model.fc1.bias

Parameter containing:
tensor([-0.0619, -0.0863,  0.0927,  0.0744, -0.0950,  0.0325,  0.0904, -0.0249,
        -0.0522, -0.0485,  0.0012,  0.0812,  0.0964,  0.0900,  0.0597,  0.0390,
         0.0722, -0.0988, -0.0109, -0.0449, -0.0343,  0.0411, -0.0950,  0.0492,
        -0.0842, -0.0188,  0.0166, -0.0883, -0.0955, -0.0525, -0.0417,  0.0552,
         0.0764,  0.0963, -0.0266,  0.0833, -0.0483,  0.0557, -0.0122,  0.0393,
         0.0739,  0.0670,  0.0118,  0.0106,  0.0481,  0.0649, -0.0343,  0.0978,
        -0.0371,  0.0994, -0.0511, -0.0754, -0.0746,  0.0816, -0.0024, -0.0008,
         0.0212, -0.0362, -0.0906, -0.0035, -0.0017,  0.0023,  0.0819, -0.0661,
         0.0009, -0.0967,  0.0849,  0.0885, -0.0795, -0.0244,  0.0587, -0.0305,
        -0.0644,  0.0339, -0.0153, -0.0351, -0.0587,  0.0389, -0.0768,  0.0721,
         0.0716, -0.0819, -0.0380,  0.0152,  0.0039, -0.0308,  0.0980, -0.0281,
         0.0098,  0.0126,  0.0306,  0.0394,  0.0318,  0.0502,  0.0271, -0.0933,
        -0.0308, -

In [ ]:
cache_w = model.fc1.weight
cache_b = model.fc1.bias
model.fc1 = LoRALinear(100, 150)
model.fc1.weight = cache_w
model.fc1.bias = cache_b

In [ ]:
model

MyModel(
  (fc1): LoRALinear()
  (fc2): Linear(in_features=150, out_features=200, bias=True)
  (fc3): Linear(in_features=200, out_features=250, bias=True)
)

In [ ]:
model.fc2.weight.requires_grad =False
model.fc2.bias.requires_grad =False
model.fc3.weight.requires_grad =False
model.fc3.bias.requires_grad =False

In [ ]:
for name, parameter in model.named_parameters():
    print(name, parameter.requires_grad)

fc1.weight True
fc1.A True
fc1.B True
fc1.bias True
fc2.weight False
fc2.bias False
fc3.weight False
fc3.bias False


In [ ]:
from peft import LoraConfig, get_peft_model

In [ ]:
lora_config = LoraConfig(r=50, target_modules=['fc1', 'fc2', 'fc3'])

In [ ]:
lora_model = get_peft_model(model, lora_config)

ValueError: Target module LoRALinear() is not supported. Currently, only the following modules are supported: `torch.nn.Linear`, `torch.nn.Embedding`, `torch.nn.Conv1d`, `torch.nn.Conv2d`, `torch.nn.Conv3d`, `transformers.pytorch_utils.Conv1D`, `torch.nn.MultiheadAttention.`.